# IDX — Colab Training Adapter (thin)

**Roles**
- GitHub Actions = daily operational signals (`ops_sma_v0`)
- **Colab = heavy/weekend ML training & research**
- Telegram = output only
- Paper portfolio = measurement only

**Safety**
- No auto-promotion
- Production pointer unchanged
- Economic edge remains **UNVERIFIED**
- GPU optional — CPU always works

Open from GitHub: use *Open in Colab* badge in `colab/README.md`.


In [ ]:
# 1) Clone / checkout (edit REF as needed)
import os, sys
REPO = os.environ.get("IDX_REPO", "https://github.com/whatman42/idx.git")
REF = os.environ.get("IDX_REF", "main")
WORKDIR = "/content/idx"
if not os.path.exists(WORKDIR):
    !git clone --depth 1 -b $REF $REPO $WORKDIR
else:
    %cd /content/idx
    !git fetch --depth 1 origin $REF
    !git checkout $REF
    !git pull --ff-only origin $REF
%cd /content/idx
sys.path.insert(0, "/content/idx")
print("cwd", os.getcwd())


In [ ]:
# 2) Install deps (CPU-safe; GPU libs optional)
!pip -q install -r requirements.txt
!pip -q install -r requirements-colab.txt


In [ ]:
# 3) Hardware probe
from src.python.colab.hardware import probe_hardware, benchmark_lgbm_cpu_gpu
hw = probe_hardware(budget_sec=float(__import__("os").getenv("COLAB_TRAINING_BUDGET_SEC", "1200")))
print(hw.print_summary())
print(hw.to_dict())


In [ ]:
# 4) Optional CPU vs GPU benchmark (skipped automatically if no GPU)
bench = None
if hw.cuda_available:
    bench = benchmark_lgbm_cpu_gpu()
    print(bench)
else:
    print({"backend": "cpu", "reason": "no_gpu"})


In [ ]:
# 5) Run governed training via repository entrypoint (NOT notebook-local ML)
import os
from src.python.colab.run_training import run_colab_training

report = run_colab_training(
    budget_sec=float(os.getenv("COLAB_TRAINING_BUDGET_SEC", "1200")),
    out_dir="models/candidates",
    report_dir="artifacts/training",
    promote=False,
    run_shadow=True,
    try_gpu_benchmark=True,
)
print("status:", report.get("status"))
print("promoted:", report.get("promoted"))
print("pointer_unchanged:", report.get("production_pointer_unchanged"))
print("edge:", report.get("economic_edge"))
print("models:", [m.get("model_id") for m in report.get("models") or []])


In [ ]:
# 6) Optional: publish reports to GitHub (requires Colab secret GH_PAT)
# Secrets → GH_PAT (repo scope). Training works WITHOUT this secret.
from src.python.colab.publish import publish_candidates
pub = publish_candidates(
    paths=[
        "artifacts/training/last_training_report.json",
        "artifacts/training/shadow_report.json",
    ]
)
print({k: v for k, v in pub.items() if k != "token"})
assert pub.get("production_pointer_touched") is False
